In [0]:
#load csv load
df = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv",
    header=True,
    inferSchema=True
)

In [0]:
#create delta path 
delta_path = "dbfs:/FileStore/ecommerce_uc/delta/events"

In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()


+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|     ecommerce_uc|         default|
+-----------------+----------------+



In [0]:
spark.sql("USE CATALOG ecommerce_uc")
spark.sql("USE SCHEMA bronze_layer")


DataFrame[]

In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()


+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|     ecommerce_uc|    bronze_layer|
+-----------------+----------------+



In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("events")


In [0]:
spark.sql("SHOW TABLES IN bronze_layer").show()


+------------+---------+-----------+
|    database|tableName|isTemporary|
+------------+---------+-----------+
|bronze_layer|   events|      false|
+------------+---------+-----------+



In [0]:
bronze_df = spark.table("workspace.bronze.events")


Silver stage

In [0]:
from pyspark.sql.functions import col, to_timestamp

silver_df = bronze_df \
    .filter(col("event_type").isNotNull()) \
    .filter(col("price").isNotNull() & (col("price") > 0)) \
    .filter(col("category_code").isNotNull()) \
    .withColumn("event_time", to_timestamp("event_time"))


In [0]:
silver_df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("ecommerce_uc.silver_layer.cleaned_events")


In [0]:
spark.sql("SELECT COUNT(*) FROM ecommerce_uc.silver_layer.cleaned_events").show()
spark.table("ecommerce_uc.silver_layer.cleaned_events").limit(5).display()


+--------+
|COUNT(*)|
+--------+
|45511887|
+--------+



event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-11-15T13:39:42.000Z,view,6200987,2053013552293216471,appliances.environment.air_heater,null,188.91,529566948,79c74259-bd2d-4c47-a08c-23b3c4e98179
2019-11-15T13:39:42.000Z,view,2702420,2053013563911439225,appliances.kitchen.refrigerators,dauscher,270.25,554869281,e44552eb-e03f-4f15-9b98-3225708517cf
2019-11-15T13:39:42.000Z,view,18300526,2053013558945383017,accessories.bag,trust,28.29,513395415,a3090ecc-c02d-4e74-b15e-9784968a19ba
2019-11-15T13:39:42.000Z,view,1201504,2172371436436455782,electronics.tablet,samsung,150.03,519038000,377672f8-0b2e-43fd-905e-30c7c99b877c
2019-11-15T13:39:42.000Z,view,3601505,2053013563810775923,appliances.kitchen.washer,samsung,474.25,513668847,98a76c3a-917a-4410-be84-a4ffae1b776f


Gold layer


In [0]:
silver_df = spark.table("ecommerce_uc.silver_layer.cleaned_events")


In [0]:
from pyspark.sql.functions import sum, count

gold_df = silver_df \
    .filter(col("event_type") == "purchase") \
    .groupBy("category_code") \
    .agg(
        sum("price").alias("total_revenue"),
        count("*").alias("total_purchases")
    )


In [0]:
gold_df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("ecommerce_uc.gold_layer.category_metrics")


In [0]:
spark.sql("""
SELECT *
FROM ecommerce_uc.gold_layer.category_metrics
ORDER BY total_revenue DESC
LIMIT 10
""").display()


category_code,total_revenue,total_purchases
electronics.smartphone,1.77821661610003E8,382647
electronics.video.tv,1.2457151160000036E7,30274
computers.notebook,1.067842971E7,18433
electronics.clocks,6552737.249999973,23237
appliances.kitchen.washer,5801906.329999979,19772
electronics.audio.headphone,5669502.489999962,40834
appliances.kitchen.refrigerators,4722657.299999999,13042
appliances.environment.vacuum,2762311.620000007,18193
computers.desktop,1556637.6999999995,3781
electronics.tablet,1520253.009999998,6138


In [0]:
#Grant read-only access
spark.sql("GRANT SELECT ON TABLE ecommerce_uc.gold_layer.category_metrics TO `r.reshma91@gmail.com`")


DataFrame[]